# Project Aura 2.0 - Notebook 07: Deployment PreparationThis notebook prepares models and configurations for cloud deployment:- Export trained models (ONNX, TorchScript formats)- Generate Dockerfile for containerized inference- Create cloud deployment configuration- Validate deployment artifactsImports reusable modules from `src/` where practical.

In [ ]:
import osimport sysimport jsonfrom pathlib import PathPROJECT_DIR = "/content/project-aura"if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)from src.core.config import load_configfrom src.utils.logging import get_loggerlogger = get_logger("notebook-07")config = load_config()print("Configuration loaded")

In [ ]:
DEPLOY_DIR = "/content/project-aura/deploy"os.makedirs(DEPLOY_DIR, exist_ok=True)print(f"Deployment directory: {DEPLOY_DIR}")

In [ ]:
import torchclass ExportModel(torch.nn.Module):    def __init__(self, num_classes):        super().__init__()        self.features = torch.nn.Sequential(            torch.nn.Conv2d(3, 16, 3, padding=1),            torch.nn.ReLU(),            torch.nn.AdaptiveAvgPool2d((1, 1)),            torch.nn.Flatten(),            torch.nn.Linear(16, num_classes),        )        def forward(self, x):        return self.features(x)export_model = ExportModel(num_classes=config.vision.num_classes)export_model.eval()dummy_input = torch.randn(1, 3, *config.vision.input_size)onnx_path = os.path.join(DEPLOY_DIR, "model.onnx")try:    torch.onnx.export(        export_model,        dummy_input,        onnx_path,        export_params=True,        opset_version=11,        do_constant_folding=True,        input_names=["input"],        output_names=["output"],    )    print(f"Model exported to ONNX: {onnx_path}")    print(f"ONNX file size: {os.path.getsize(onnx_path)} bytes")except Exception as e:    print(f"ONNX export failed: {e}")    print("Continuing without ONNX export")

In [ ]:
torchscript_path = os.path.join(DEPLOY_DIR, "model.pt")try:    traced = torch.jit.trace(export_model, dummy_input)    traced.save(torchscript_path)    print(f"Model exported to TorchScript: {torchscript_path}")    print(f"TorchScript file size: {os.path.getsize(torchscript_path)} bytes")except Exception as e:    print(f"TorchScript export failed: {e}")

In [ ]:
dockerfile_content = "FROM python:3.11-slim\n\nWORKDIR /app\n\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY src/ ./src/\n\nENV PYTHONUNBUFFERED=1\nEXPOSE 8000\n\nCMD ["python", "-m", "uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]\n"dockerfile_path = os.path.join(DEPLOY_DIR, "Dockerfile")with open(dockerfile_path, "w") as f:    f.write(dockerfile_content)print(f"Dockerfile generated: {dockerfile_path}")

In [ ]:
requirements = [    "fastapi>=0.100.0",    "uvicorn>=0.23.0",    "numpy>=1.24.0",    "pillow>=10.0.0",    "torch>=2.0.0",    "torchvision>=0.15.0",    "google-cloud-storage>=2.10.0",    "pydantic>=2.0.0",]requirements_path = os.path.join(DEPLOY_DIR, "requirements.txt")with open(requirements_path, "w") as f:    f.write("\n".join(requirements) + "\n")print(f"Requirements generated: {requirements_path}")print("\nDependencies:")for req in requirements:    print(f"  {req}")

In [ ]:
deploy_config = {    "project": {        "name": "project-aura",        "version": "2.0.0",        "provider": os.getenv("CLOUD_PROVIDER", "gcp"),        "region": os.getenv("GCP_REGION", "us-central1"),    },    "inference": {        "model_name": config.vision.model_name,        "num_classes": config.vision.num_classes,        "input_size": list(config.vision.input_size),        "batch_size": config.vision.batch_size,    },    "deployment": {        "min_instances": 1,        "max_instances": 10,        "cpu": "1",        "memory": "512Mi",        "concurrency": 80,    },    "cloud_storage": {        "bucket_name": os.getenv("GCS_BUCKET_NAME", ""),        "project_id": os.getenv("GCP_PROJECT_ID", ""),    },}config_path = os.path.join(DEPLOY_DIR, "deploy_config.json")with open(config_path, "w") as f:    json.dump(deploy_config, f, indent=2)print(f"Deployment config saved: {config_path}")print(json.dumps(deploy_config, indent=2))

In [ ]:
artifacts = [    (dockerfile_path, "Dockerfile"),    (requirements_path, "requirements.txt"),    (config_path, "deploy_config.json"),]if os.path.exists(onnx_path):    artifacts.append((onnx_path, "model.onnx"))if os.path.exists(torchscript_path):    artifacts.append((torchscript_path, "model.pt"))print("\nDeployment artifacts:")all_valid = Truefor path, name in artifacts:    exists = os.path.exists(path)    size = os.path.getsize(path) if exists else 0    status = "VALID" if exists else "MISSING"    if not exists:        all_valid = False    print(f"  {name}: {status} ({size} bytes)")print("\nAll artifacts valid:", all_valid)print("\nNext: Run 08_monitoring_analytics.ipynb for monitoring setup")